# Model Context Protocol (MCP): From Basics to Advanced

## 📚 Learning Objectives
- Understand the concept of **Function Calling** / **Tool Calling** from the ground up
- Master real-world issues when using function calling
- Understand what **Model Context Protocol (MCP)** is and why we need it
- Write your first **MCP Server**
- Connect **LLM** with **MCP Server**
- Integrate **MCP** into **LangChain** and **LangGraph**

---

## 🎯 Notebook Structure
1. **Function Calling (Tool Calling) Basics**
2. **Real-World Problems - Why do we need MCP?**
3. **What is MCP? Architecture and Concepts**
4. **Creating a Simple MCP Server**
5. **Connecting LLM with MCP Server**
6. **MCP + LangChain**
7. **MCP + LangGraph (Agentic Chatbot)**

---

# PART 1: Function Calling (Tool Calling) - Basics from Scratch

## 1.1 Concept of Function Calling / Tool Calling

**Function Calling** (also known as **Tool Calling**) is the ability of an LLM to:
1. **Recognize** that a tool needs to be called
2. **Select** the appropriate tool from available tools
3. **Extract parameters** from the user's prompt
4. **Execute** the tool with those parameters
5. **Return results** to the LLM for further processing

### Real-World Example:
```
User: "What's the weather in Hanoi today?"
LLM will:
  → Recognize need to use tool: get_weather
  → Extract parameter: city="Hanoi"
  → Call function get_weather(city="Hanoi")
  → Receive result: "20°C, Cloudy"
  → Reply to user: "Today in Hanoi it's 20°C and cloudy..."
```

---

## 1.2 Manual Approach: Building Function Calling from Scratch

In [1]:
# STEP 1: Define tools
import json
from typing import Any, Callable, Dict

# Tool 1: Simple Calculator
def calculator(operation: str, a: float, b: float) -> float:
    """
    Perform basic mathematical operations
    Args:
        operation: 'add', 'subtract', 'multiply', 'divide'
        a, b: numbers to calculate
    """
    if operation == "add":
        return a + b
    elif operation == "subtract":
        return a - b
    elif operation == "multiply":
        return a * b
    elif operation == "divide":
        if b == 0:
            raise ValueError("Cannot divide by zero")
        return a / b
    else:
        raise ValueError(f"Invalid operation: {operation}")

# Tool 2: Get weather information (mock)
def get_weather(city: str) -> Dict[str, Any]:
    """Get weather information for a city"""
    weather_data = {
        "Hanoi": {"temp": 20, "condition": "Cloudy", "humidity": 75},
        "Ho Chi Minh": {"temp": 28, "condition": "Sunny", "humidity": 80},
        "Da Nang": {"temp": 25, "condition": "Partly Cloudy", "humidity": 78},
    }
    if city not in weather_data:
        return {"error": f"No weather data available for {city}"}
    return weather_data[city]

# Tool 3: Search information (mock)
def search_knowledge_base(query: str) -> str:
    """Search information from knowledge base"""
    knowledge = {
        "python": "Python is a powerful and easy-to-learn programming language",
        "machine learning": "Machine Learning is an AI field focused on learning from data",
        "llm": "LLM (Large Language Model) refers to large language models like GPT, Claude",
    }
    query_lower = query.lower()
    for key, value in knowledge.items():
        if key in query_lower:
            return value
    return f"No information found about: {query}"

print("✅ Defined 3 tools: calculator, get_weather, search_knowledge_base")

✅ Defined 3 tools: calculator, get_weather, search_knowledge_base


In [2]:
# STEP 2: Create Registry to manage tools
class ToolRegistry:
    """
    Manage all available tools
    Like a "lookup tool" to find the function to call
    """
    def __init__(self):
        self.tools: Dict[str, Dict[str, Any]] = {}
    
    def register(self, name: str, func: Callable, description: str, parameters: Dict):
        """
        Register a tool
        Args:
            name: tool name (used by LLM to call it)
            func: actual Python function
            description: tool description (tells LLM what it does)
            parameters: parameter schema
        """
        self.tools[name] = {
            "func": func,
            "description": description,
            "parameters": parameters
        }
    
    def get_tool_schema(self) -> str:
        """
        Return schema of all tools (sent to LLM)
        LLM uses this schema to know which tools are available
        """
        schema = []
        for name, tool_info in self.tools.items():
            schema.append({
                "name": name,
                "description": tool_info["description"],
                "parameters": tool_info["parameters"]
            })
        return json.dumps(schema, indent=2)
    
    def call_tool(self, tool_name: str, **kwargs) -> Any:
        """
        Call a tool by name
        """
        if tool_name not in self.tools:
            raise ValueError(f"Tool does not exist: {tool_name}")
        
        func = self.tools[tool_name]["func"]
        return func(**kwargs)

# STEP 3: Register tools in registry
registry = ToolRegistry()

# Register tool: calculator
registry.register(
    name="calculator",
    func=calculator,
    description="Perform basic math operations (add, subtract, multiply, divide)",
    parameters={
        "type": "object",
        "properties": {
            "operation": {
                "type": "string",
                "enum": ["add", "subtract", "multiply", "divide"],
                "description": "The operation to perform"
            },
            "a": {
                "type": "number",
                "description": "First number"
            },
            "b": {
                "type": "number",
                "description": "Second number"
            }
        },
        "required": ["operation", "a", "b"]
    }
)

# Register tool: get_weather
registry.register(
    name="get_weather",
    func=get_weather,
    description="Get weather information for a city",
    parameters={
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "City name"
            }
        },
        "required": ["city"]
    }
)

# Register tool: search_knowledge_base
registry.register(
    name="search_knowledge_base",
    func=search_knowledge_base,
    description="Search information from knowledge base",
    parameters={
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search keyword"
            }
        },
        "required": ["query"]
    }
)

print("✅ Registered 3 tools in registry")
print("\n📋 Schema of all tools:")
print(registry.get_tool_schema())

✅ Registered 3 tools in registry

📋 Schema of all tools:
[
  {
    "name": "calculator",
    "description": "Perform basic math operations (add, subtract, multiply, divide)",
    "parameters": {
      "type": "object",
      "properties": {
        "operation": {
          "type": "string",
          "enum": [
            "add",
            "subtract",
            "multiply",
            "divide"
          ],
          "description": "The operation to perform"
        },
        "a": {
          "type": "number",
          "description": "First number"
        },
        "b": {
          "type": "number",
          "description": "Second number"
        }
      },
      "required": [
        "operation",
        "a",
        "b"
      ]
    }
  },
  {
    "name": "get_weather",
    "description": "Get weather information for a city",
    "parameters": {
      "type": "object",
      "properties": {
        "city": {
          "type": "string",
          "description": "City name"
     

In [3]:
# STEP 4: Simulate LLM Tool Call Decision
# (In reality, we'd use OpenAI, Claude API, but here we mock it)

class MockLLMResponse:
    """
    Simulate LLM response
    In reality, LLM would analyze user input and decide which tool to call
    """
    def __init__(self, tool_name: str, parameters: Dict):
        self.tool_name = tool_name
        self.parameters = parameters
        self.reasoning = f"Call tool '{tool_name}' with parameters {parameters}"

# Simulate: LLM receives user input and decides which tool to call
def mock_llm_decision(user_input: str) -> MockLLMResponse:
    """
    Simulate LLM analyzing user input and deciding which tool to call
    In reality, this would call OpenAI/Claude API
    """
    user_input_lower = user_input.lower()
    
    # Heuristics (rules) to decide tool
    if "add" in user_input_lower or "plus" in user_input_lower:
        # User wants to calculate 5 + 3
        if "5" in user_input and "3" in user_input:
            return MockLLMResponse("calculator", {"operation": "add", "a": 5, "b": 3})
    
    if "weather" in user_input_lower and "hanoi" in user_input_lower:
        return MockLLMResponse("get_weather", {"city": "Hanoi"})
    
    if "weather" in user_input_lower and "ho chi minh" in user_input_lower:
        return MockLLMResponse("get_weather", {"city": "Ho Chi Minh"})
    
    if "python" in user_input_lower or "machine learning" in user_input_lower:
        return MockLLMResponse("search_knowledge_base", {"query": user_input})
    
    return None

# STEP 5: Function Calling Loop - Tool execution loop
def function_calling_loop(user_input: str) -> str:
    """
    Function calling loop:
    1. LLM receives input
    2. LLM decides which tool to call
    3. System calls the tool
    4. LLM processes the result
    """
    print(f"👤 User: {user_input}")
    print("-" * 60)
    
    # Step 1: LLM receives input and makes decision
    llm_decision = mock_llm_decision(user_input)
    
    if llm_decision is None:
        return "Sorry, I don't know how to help you."
    
    # Step 2: Execute tool call
    print(f"🤖 LLM Decision:")
    print(f"   Tool: {llm_decision.tool_name}")
    print(f"   Parameters: {llm_decision.parameters}")
    print()
    
    try:
        # Call tool from registry
        result = registry.call_tool(llm_decision.tool_name, **llm_decision.parameters)
        print(f"🔧 Tool Execution Result:")
        print(f"   {json.dumps(result, indent=4, ensure_ascii=False)}")
        print()
        
        # Step 3: LLM processes result and replies to user
        response = format_llm_response(llm_decision.tool_name, result, user_input)
        print(f"💬 Assistant: {response}")
        return response
    
    except Exception as e:
        error_msg = f"Error executing tool: {str(e)}"
        print(f"❌ Error: {error_msg}")
        return error_msg

def format_llm_response(tool_name: str, result: Any, user_input: str) -> str:
    """Format result into a reply for the user"""
    if tool_name == "calculator":
        return f"The result is: {result}"
    
    elif tool_name == "get_weather":
        if "error" in result:
            return result["error"]
        return f"The weather is {result['condition']}, temperature {result['temp']}°C, humidity {result['humidity']}%"
    
    elif tool_name == "search_knowledge_base":
        return f"Found: {result}"
    
    return f"Result: {result}"

print("✅ Function Calling Loop has been defined")

✅ Function Calling Loop has been defined


In [4]:
# BƯỚC 6: Test Function Calling
print("=" * 60)
print("TEST 1: Calculation")
print("=" * 60)
function_calling_loop("What is 5 plus 3?")

print("\n")
print("=" * 60)
print("TEST 2: Ask about weather")
print("=" * 60)
function_calling_loop("What's the weather in Hanoi today?")

print("\n")
print("=" * 60)
print("TEST 3: Search information")
print("=" * 60)
function_calling_loop("What is Python?")

TEST 1: Calculation
👤 User: What is 5 plus 3?
------------------------------------------------------------
🤖 LLM Decision:
   Tool: calculator
   Parameters: {'operation': 'add', 'a': 5, 'b': 3}

🔧 Tool Execution Result:
   8

💬 Assistant: The result is: 8


TEST 2: Ask about weather
👤 User: What's the weather in Hanoi today?
------------------------------------------------------------
🤖 LLM Decision:
   Tool: get_weather
   Parameters: {'city': 'Hanoi'}

🔧 Tool Execution Result:
   {
    "temp": 20,
    "condition": "Cloudy",
    "humidity": 75
}

💬 Assistant: The weather is Cloudy, temperature 20°C, humidity 75%


TEST 3: Search information
👤 User: What is Python?
------------------------------------------------------------
🤖 LLM Decision:
   Tool: search_knowledge_base
   Parameters: {'query': 'What is Python?'}

🔧 Tool Execution Result:
   "Python is a powerful and easy-to-learn programming language"

💬 Assistant: Found: Python is a powerful and easy-to-learn programming language


'Found: Python is a powerful and easy-to-learn programming language'

## 1.3 Function Calling with OpenAI API (Professional Approach)

The approach above is to understand the logic. In reality, we use the OpenAI Function Calling API:

### Benefits of OpenAI Function Calling:
- **Smarter LLM**: OpenAI API automatically analyzes user input to decide which tool to call
- **Better parameter handling**: API extracts parameters precisely from text
- **Error handling**: API handles edge cases
- **Streaming**: Supports real-time streaming responses

### OpenAI Function Calling Structure:

In [5]:
# Ví dụ cấu trúc OpenAI Function Calling
# (No API key needed to see the structure)

openai_tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Perform basic math operations (add, subtract, multiply, divide)",
            "parameters": {
                "type": "object",
                "properties": {
                    "operation": {
                        "type": "string",
                        "enum": ["add", "subtract", "multiply", "divide"],
                        "description": "The operation to perform"
                    },
                    "a": {
                        "type": "number",
                        "description": "First number"
                    },
                    "b": {
                        "type": "number",
                        "description": "Second number"
                    }
                },
                "required": ["operation", "a", "b"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get weather information for a city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "City name"
                    }
                },
                "required": ["city"]
            }
        }
    }
]

print("📋 OpenAI Function Schema:")
print(json.dumps(openai_tools_schema, indent=2, ensure_ascii=False))

print("\n✅ Benefits of OpenAI Function Calling vs manual approach:")
print("""
┌─────────────────────────┬────────────────────┬─────────────────────────┐
│ Criteria                │ Manual Approach    │ OpenAI Function Calling │
├─────────────────────────┼────────────────────┼─────────────────────────┤
│ Extract parameters      │ Use regex/rules    │ Smart, context-aware    │
│ Error handling          │ Manual coding      │ Automatic               │
│ Understand intent       │ Simple heuristics  │ Language model          │
│ Extensibility           │ Hard to add tools  │ Easy to add new tools   │
│ Performance             │ Fast but limited   │ Slower but smarter      │
└─────────────────────────┴────────────────────┴─────────────────────────┘
""")

📋 OpenAI Function Schema:
[
  {
    "type": "function",
    "function": {
      "name": "calculator",
      "description": "Perform basic math operations (add, subtract, multiply, divide)",
      "parameters": {
        "type": "object",
        "properties": {
          "operation": {
            "type": "string",
            "enum": [
              "add",
              "subtract",
              "multiply",
              "divide"
            ],
            "description": "The operation to perform"
          },
          "a": {
            "type": "number",
            "description": "First number"
          },
          "b": {
            "type": "number",
            "description": "Second number"
          }
        },
        "required": [
          "operation",
          "a",
          "b"
        ]
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "get_weather",
      "description": "Get weather information for a city",
      "parameters": {
        "

---

# PART 2: Real-World Problems - Why Do We Need MCP?

## 2.1 Các Vấn Đề Khi Sử Dụng Function Calling Thông Thường

### Vấn Đề 1: Nhiều LLM, Nhiều Format Khác Nhau
```
OpenAI API:
{
  "type": "function",
  "function": { "name": "...", "parameters": ... }
}

Claude API:
{
  "name": "tool_name",
  "description": "...",
  "input_schema": { ... }
}

Gemini API:
{
  "function_declarations": [
    { "name": "...", "description": "...", "parameters": ... }
  ]
}
```

**Vấn đề**: Mỗi LLM provider có format khác → Phải code riêng cho từng LLM

### Vấn Đề 2: Nhiều Tools, Khó Quản Lý
- System A cần: calculator, weather, search
- System B cần: calendar, email, database_query
- System C cần: calculator, weather, calendar, email, database_query

**Vấn đề**: Nhiều tools được copy-paste, khó maintain, dễ inconsistent

### Vấn Đề 3: Tool Server Vs LLM Coupling
```
Cách thủ công:
┌──────────────┐      request      ┌──────────────┐
│   LLM        │ ─────────────────→ │   Tool       │
│   Client     │ ←───────────────── │   Server     │
└──────────────┘      response      └──────────────┘

Problem:
- LLM cần biết chi tiết của tool
- Thay đổi tool → phải update LLM code
- Khó scale khi có nhiều tool servers
```

### Vấn Đề 4: Không Có Giao dayc Chung
- Mỗi company tự implement cách của riêng
- Không interoperability
- Khó tích hợp tools từ các sources khác nhau

## 2.2 Giải Pháp: Model Context Protocol (MCP)


```
With MCP:
┌──────────────┐                    ┌──────────────┐
│   LLM        │  MCP Protocol      │   MCP Server │
│   Client     │ ──────────────────→│   (Tools)    │
│              │ ←────────────────── │              │
└──────────────┘                    └──────────────┘

Benefits:
✅ LLM doesn't need to know implementation details of the tool
✅ Common protocol for all LLM providers
✅ Easy to add/remove/swap tools without changing LLM
✅ Can reuse MCP servers between applications
✅ Standardized, official support from Anthropic
```

---

# PART 3: What is MCP? Architecture and Concepts

## 3.1 Definition of MCP

**Model Context Protocol (MCP)** is a standard protocol developed by Anthropic to:
1. **Connect LLM** with external tools/resources
2. **Standardize** communication between LLM clients and tool servers
3. **Decouple** LLM logic from tool logic

**Similar to:**
- HTTP is the protocol for web browsers to connect with web servers
- MCP is the protocol for LLM clients to connect with tool servers

## 3.2 MCP Architecture - Architecture

In [6]:
# Vẽ sơ đồ kiến trúc MCP
from typing import List

mcp_architecture = """
┌───────────────────────────────────────────────────────────────────┐
│                    MCP Architecture Diagram                        │
└───────────────────────────────────────────────────────────────────┘

1. HIGH LEVEL VIEW:
┌──────────────────────────────────────────────────────────────────┐
│                         MCP Client                                 │
│  (Claude API / OpenAI / LangChain / LangGraph)                    │
└────────────────────────────┬─────────────────────────────────────┘
                             │
                     MCP Protocol (JSON-RPC)
                             │
        ┌────────────────────┼────────────────────┐
        │                    │                    │
┌───────▼──────────┐ ┌──────▼──────────┐ ┌──────▼──────────┐
│  MCP Server 1    │ │  MCP Server 2    │ │  MCP Server 3  │
│  (Calculator)    │ │  (Weather)       │ │  (Database)    │
└──────────────────┘ └──────────────────┘ └──────────────────┘


2. DETAILED MESSAGE FLOW:

Client                                          Server
  │                                              │
  ├─────── Initialize Request ────────────────→ │
  │        {type: "initialize", version: ...}  │
  │                                              │
  │ ←────── Initialize Response ──────────────┤ │
  │         {version: ..., capabilities: ...}  │
  │                                              │
  │                                              │
  │  [Client sends tools list request]          │
  ├───── ListTools Request ───────────────────→ │
  │                                              │
  │ ←──── ListTools Response ──────────────────┤ │
  │       [{name: "calculator", schema: ...},   │
  │        {name: "weather", schema: ...}]     │
  │                                              │
  │                                              │
  │  [User asks something, LLM decides to use a tool]
  │  [Client calls the tool]                    │
  ├────── CallTool Request ───────────────────→ │
  │       {tool: "calculator", args: {...}}    │
  │                                              │
  │ ←────── CallTool Response ─────────────────┤ │
  │         {result: "8"}                       │
  │                                              │


3. KEY COMPONENTS:

┌─────────────────────────────────┐
│      MCP Client                 │
├─────────────────────────────────┤
│ • LLM Integration               │
│ • Tool Calling Logic            │
│ • Response Formatting           │
│ • Error Handling                │
└────────────────┬────────────────┘
             │
    ┌────────┴────────┐
    │  MCP Protocol   │
    │  (JSON-RPC)     │
    └────────┬────────┘
             │
┌────────────▼────────────────────┐
│      MCP Server                 │
├─────────────────────────────────┤
│ • Tool Registry                 │
│ • Tool Execution                │
│ • Resource Management           │
│ • Prompt Helpers                │
└─────────────────────────────────┘


4. MCP SERVER CAPABILITIES:

┌─────────────────────────────────┐
│    MCP Server Features          │
├─────────────────────────────────┤
│ 1. Tools                        │
│    - Executable functions       │
│    - Takes input, returns output│
│    - Similar to function calling│
│                                 │
│ 2. Resources                    │
│    - Read-only data             │
│    - Files, templates, etc.     │
│                                 │
│ 3. Prompts                      │
│    - Reusable prompt templates  │
│    - System prompts, examples   │
│                                 │
│ 4. Sampling                     │
│    - Delegate inference to LLM  │
│                                 │
└─────────────────────────────────┘
"""

print(mcp_architecture)


┌───────────────────────────────────────────────────────────────────┐
│                    MCP Architecture Diagram                        │
└───────────────────────────────────────────────────────────────────┘

1. HIGH LEVEL VIEW:
┌──────────────────────────────────────────────────────────────────┐
│                         MCP Client                                 │
│  (Claude API / OpenAI / LangChain / LangGraph)                    │
└────────────────────────────┬─────────────────────────────────────┘
                             │
                     MCP Protocol (JSON-RPC)
                             │
        ┌────────────────────┼────────────────────┐
        │                    │                    │
┌───────▼──────────┐ ┌──────▼──────────┐ ┌──────▼──────────┐
│  MCP Server 1    │ │  MCP Server 2    │ │  MCP Server 3  │
│  (Calculator)    │ │  (Weather)       │ │  (Database)    │
└──────────────────┘ └──────────────────┘ └──────────────────┘


2. DETAILED MESSAGE FLOW:

Cl

## 3.3 MCP vs Function Calling - So Sánh Chi Tiết

| Đặc Tính | Function Calling | MCP |
|---------|------------------|-----|
| **Giao thức** | Riêng cho từng LLM (OpenAI, Claude, etc.) | Unified protocol |
| **Standardization** | Non-standard | Official standard (Anthropic) |
| **Tool Discovery** | LLM cần biết tools trước | Dynamic tool discovery |
| **Decoupling** | LLM và Tools coupling cao | Loose coupling |
| **Scalability** | Khó scale nhiều tools | Dễ scale |
| **Reusability** | Tools khó reuse giữa apps | Tools dễ reuse |
| **Error Handling** | Phải implement riêng | Built-in handling |

## 3.4 MCP Message Protocol - JSON-RPC

MCP sử dụng **JSON-RPC 2.0** để communicate:

```json
// Request từ Client
{
  "jsonrpc": "2.0",
  "id": 1,
  "method": "tools/call",
  "params": {
    "name": "calculator",
    "arguments": {"operation": "add", "a": 5, "b": 3}
  }
}

// Response từ Server
{
  "jsonrpc": "2.0",
  "id": 1,
  "result": {
    "content": [{"type": "text", "text": "8"}]
  }
}
```

---

# PART 4: Creating a Simple MCP Server from Scratch

## 4.1 Giới Thiệu MCP SDK

Anthropic cung cấp **mcp** library Python để làm việc với MCP.

**Installation:**
```bash
pip install mcp
```

**Khái niệm chính:**
- **Server**: Implements MCP protocol, exposes tools
- **Client**: Connects to server, calls tools
- **Tools**: Functions that can be called
- **Resources**: Data that can be read
- **Prompts**: Template prompts

In [6]:
# Check if the mcp library is available
try:
    import mcp
    print("✅ MCP library is already installed")
except ImportError:
    print("⚠️ MCP library is not installed, installing...")
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "mcp"])
    import mcp
    print("✅ MCP library installed successfully")

✅ MCP library is already installed


## 4.2 Tạo MCP Server Đơn Giản - Calculator Server

In [7]:
from mcp.server import Server
from mcp.types import Tool, TextContent

# Step 1: Create MCP Server instance
server = Server("calculator-server")

# Step 2: Define tool handlers
@server.call_tool()
async def call_calculator(name: str, arguments: dict):
    """
    Handler for tool call
    Called when client calls a tool
    """
    if name == "add":
        result = arguments["a"] + arguments["b"]
        return [TextContent(type="text", text=str(result))]
    
    elif name == "subtract":
        result = arguments["a"] - arguments["b"]
        return [TextContent(type="text", text=str(result))]
    
    elif name == "multiply":
        result = arguments["a"] * arguments["b"]
        return [TextContent(type="text", text=str(result))]
    
    elif name == "divide":
        if arguments["b"] == 0:
            return [TextContent(type="text", text="Error: Cannot divide by zero")]
        result = arguments["a"] / arguments["b"]
        return [TextContent(type="text", text=str(result))]
    
    else:
        return [TextContent(type="text", text=f"Unknown tool: {name}")]

# Step 3: Register tools (Tool Discovery)
@server.list_tools()
async def list_tools():
    """
    Return the list of available tools
    Client will call this function to know which tools can be used
    """
    return [
        Tool(
            name="add",
            description="Add two numbers",
            inputSchema={
                "type": "object",
                "properties": {
                    "a": {"type": "number", "description": "First number"},
                    "b": {"type": "number", "description": "Second number"}
                },
                "required": ["a", "b"]
            }
        ),
        Tool(
            name="subtract",
            description="Subtract two numbers (a - b)",
            inputSchema={
                "type": "object",
                "properties": {
                    "a": {"type": "number", "description": "Minuend"},
                    "b": {"type": "number", "description": "Subtrahend"}
                },
                "required": ["a", "b"]
            }
        ),
        Tool(
            name="multiply",
            description="Multiply two numbers",
            inputSchema={
                "type": "object",
                "properties": {
                    "a": {"type": "number", "description": "First number"},
                    "b": {"type": "number", "description": "Second number"}
                },
                "required": ["a", "b"]
            }
        ),
        Tool(
            name="divide",
            description="Divide two numbers (a / b)",
            inputSchema={
                "type": "object",
                "properties": {
                    "a": {"type": "number", "description": "Dividend"},
                    "b": {"type": "number", "description": "Divisor"}
                },
                "required": ["a", "b"]
            }
        ),
    ]

print("✅ MCP Server created successfully")
print("📋 Tools have been defined:")
print("   - add: Add two numbers")
print("   - subtract: Subtract two numbers")
print("   - multiply: Multiply two numbers")
print("   - divide: Divide two numbers")

✅ MCP Server created successfully
📋 Tools have been defined:
   - add: Add two numbers
   - subtract: Subtract two numbers
   - multiply: Multiply two numbers
   - divide: Divide two numbers


## 4.3 Tạo MCP Weather Server - Ví dụ Phức Tạp Hơn

In [8]:
from mcp.server import Server
from mcp.types import Tool, TextContent, Resource, ResourceTemplate
import json

# Create Weather MCP Server
weather_server = Server("weather-server")

# Mock weather database
weather_database = {
    "Hanoi": {
        "temperature": 20,
        "condition": "Cloudy",
        "humidity": 75,
        "wind_speed": 10,
        "description": "Today in Hanoi is a cloudy day, temperature 20°C"
    },
    "Ho Chi Minh": {
        "temperature": 28,
        "condition": "Sunny",
        "humidity": 80,
        "wind_speed": 5,
        "description": "Today in Ho Chi Minh is a sunny day, temperature 28°C"
    },
    "Da Nang": {
        "temperature": 25,
        "condition": "Partly Cloudy",
        "humidity": 78,
        "wind_speed": 15,
        "description": "Today in Da Nang is partly cloudy with some sun, temperature 25°C"
    },
    "Can Tho": {
        "temperature": 26,
        "condition": "Rainy",
        "humidity": 85,
        "wind_speed": 8,
        "description": "Today in Can Tho is rainy, temperature 26°C"
    }
}

# Tool 1: Get current weather for a specific city
@weather_server.call_tool()
async def call_weather_tool(name: str, arguments: dict):
    if name == "get_weather":
        city = arguments.get("city", "").title()
        if city in weather_database:
            data = weather_database[city]
            result = f"Temperature: {data['temperature']}°C, Condition: {data['condition']}, Humidity: {data['humidity']}%"
            return [TextContent(type="text", text=result)]
        else:
            return [TextContent(type="text", text=f"Weather data for {city} not available")]
    
    elif name == "get_weather_forecast":
        city = arguments.get("city", "").title()
        days = arguments.get("days", 1)
        if city in weather_database:
            current = weather_database[city]
            forecast = f"Forecast for {city} ({days} day(s)): {current['description']}"
            return [TextContent(type="text", text=forecast)]
        else:
            return [TextContent(type="text", text=f"Forecast data for {city} not available")]
    
    elif name == "list_cities":
        cities = list(weather_database.keys())
        return [TextContent(type="text", text=f"Available cities: {', '.join(cities)}")]
    
    else:
        return [TextContent(type="text", text=f"Unknown tool: {name}")]

# Register tools
@weather_server.list_tools()
async def list_weather_tools():
    return [
        Tool(
            name="get_weather",
            description="Get current weather information for a specific city",
            inputSchema={
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "City name (e.g., Hanoi, Ho Chi Minh, Da Nang)"
                    }
                },
                "required": ["city"]
            }
        ),
        Tool(
            name="get_weather_forecast",
            description="Get weather forecast for upcoming days",
            inputSchema={
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "City name"
                    },
                    "days": {
                        "type": "integer",
                        "description": "Number of forecast days (default: 1)",
                        "default": 1
                    }
                },
                "required": ["city"]
            }
        ),
        Tool(
            name="list_cities",
            description="Get list of cities with available weather information",
            inputSchema={
                "type": "object",
                "properties": {}
            }
        ),
    ]

# Resources: Read-only data
@weather_server.list_resources()
async def list_weather_resources():
    """
    Resources are read-only data that clients can access.
    Unlike Tools, which require execution.
    """
    cities = list(weather_database.keys())
    return [
        Resource(
            uri=f"weather://city/{city.lower()}",
            name=f"Current weather in {city}",
            description=f"Real-time weather data for {city}",
            mimeType="application/json"
        ) for city in cities
    ]

@weather_server.read_resource()
async def read_weather_resource(uri: str):
    """Read weather data from resource URI"""
    # Example: weather://city/hanoi
    if uri.startswith("weather://city/"):
        city_name = uri.replace("weather://city/", "").title()
        if city_name in weather_database:
            data = weather_database[city_name]
            return json.dumps(data, indent=2)
    return "Resource not found"

print("✅ Weather MCP Server created successfully")
print("📋 Tools:")
print("   - get_weather: Get current weather")
print("   - get_weather_forecast: Weather forecast")
print("   - list_cities: List of cities")
print("📦 Resources registered for all cities")

✅ Weather MCP Server created successfully
📋 Tools:
   - get_weather: Get current weather
   - get_weather_forecast: Weather forecast
   - list_cities: List of cities
📦 Resources registered for all cities


---

# PART 5: Connecting LLM with MCP Server

## 5.1 MCP Client - Kết Nối với Server

In [ ]:
# 5.1 Tạo MCP Client để kết nối với Server
import asyncio
from mcp.client import ClientSession
from mcp.client.stdio import StdioClientTransport

# Giả sử MCP Server run dưới dạng stdio transport
# (Thực tế sẽ là subprocess or network)

class MCPClientExample:
    """
    Ví dụ về MCP Client
    Trong in practice, Claude API or integration với LangChain sẽ handle part này
    """
    
    def __init__(self):
        self.session = None
        self.tools = []
    
    async def initialize(self):
        """Khởi create kết nối với MCP Server"""
        # Trong notebook này, ta chỉ demo logic
        # Thực tế sẽ cần subprocess or network connection
        print("🔌 Kết nối đến MCP Server...")
    
    async def list_available_tools(self):
        """Lấy danh sách tools từ server"""
        print("📋 Lấy danh sách tools từ server...")
        # Demo: simulate get tools từ calculator server
        demo_tools = [
            {"name": "add", "description": "Cộng hai số"},
            {"name": "subtract", "description": "Trừ hai số"},
            {"name": "multiply", "description": "Nhân hai số"},
            {"name": "divide", "description": "Chia hai số"},
        ]
        self.tools = demo_tools
        return demo_tools
    
    async def call_tool(self, tool_name: str, arguments: dict):
        """Gọi one tool từ server"""
        print(f"🔧 Gọi tool: {tool_name} với arguments: {arguments}")
        # Demo result
        if tool_name == "add":
            return arguments["a"] + arguments["b"]
        elif tool_name == "subtract":
            return arguments["a"] - arguments["b"]
        elif tool_name == "multiply":
            return arguments["a"] * arguments["b"]
        elif tool_name == "divide":
            if arguments["b"] == 0:
                return "Error: Division by zero"
            return arguments["a"] / arguments["b"]

# Demo: MCP Client usage
print("✅ MCP Client class được create")
print("""
Điểm quan trọng về MCP Client:
1. Client kết nối đến MCP Server (qua stdio, network, or process)
2. Client gửi request ListTools để get danh sách tools
3. Khi LLM quyết định gọi tool, Client gửi CallTool request
4. Server xử lý và trả kết quả
5. Client trả kết quả cho LLM
""")

# Ví dụ về workflow
print("\n📊 MCP Request-Response Flow:")
print("""
┌────────────────────────────────────┐
│  1. Client: ListTools Request      │
├────────────────────────────────────┤
│  {                                  │
│    "jsonrpc": "2.0",                │
│    "method": "tools/list",          │
│    "id": 1                          │
│  }                                  │
└────────────────────────────────────┘
                 │
                 ▼
┌────────────────────────────────────┐
│  2. Server Response: [add, subtract,│
│     multiply, divide]              │
└────────────────────────────────────┘
                 │
                 ▼
┌────────────────────────────────────┐
│  3. Client: CallTool Request       │
├────────────────────────────────────┤
│  {                                  │
│    "jsonrpc": "2.0",                │
│    "method": "tools/call",          │
│    "id": 2,                         │
│    "params": {                      │
│      "name": "add",                 │
│      "arguments": {"a": 5, "b": 3}  │
│    }                                │
│  }                                  │
└────────────────────────────────────┘
                 │
                 ▼
┌────────────────────────────────────┐
│  4. Server: CallTool Response      │
├────────────────────────────────────┤
│  {                                  │
│    "jsonrpc": "2.0",                │
│    "id": 2,                         │
│    "result": {                      │
│      "content": [{                  │
│        "type": "text",              │
│        "text": "8"                  │
│      }]                             │
│    }                                │
│  }                                  │
└────────────────────────────────────┘
""")

---

# PART 6: MCP + LangChain - Framework Integration

## 6.1 LangChain + MCP Integration

LangChain cung cấp tích hợp với MCP thông qua **MCPToolkit**

**Workflow:**
```
LangChain Agent
    ↓
Cần gọi tool
    ↓
MCP Client (integrated in LangChain)
    ↓
MCP Server
    ↓
Kết quả
```

**Installation:**
```bash
pip install langchain-core mcp
```

In [10]:
# Kiểm tra và cài đặt dependencies
try:
    from langchain.agents import initialize_agent, AgentType
    from langchain_core.tools import Tool, BaseTool
    print("✅ LangChain dependencies đã cài đặt")
except ImportError:
    print("⚠️  Cài đặt LangChain dependencies...")
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "langchain", "langchain-core"])
    print("✅ Installation successful")

⚠️  Cài đặt LangChain dependencies...
✅ Installation successful


## 6.2 Tạo LangChain Agent với MCP Tools

In [11]:
from langchain_core.tools import Tool, StructuredTool
from typing import Any

# Định nghĩa các tools cho LangChain
def calculator_tool(operation: str, a: float, b: float) -> float:
    """Perform basic mathematical operations"""
    if operation == "add":
        return a + b
    elif operation == "subtract":
        return a - b
    elif operation == "multiply":
        return a * b
    elif operation == "divide":
        if b == 0:
            return None
        return a / b

def weather_tool(city: str) -> str:
    """Lấy thông tin thời tiết"""
    weather_data = {
        "Hanoi": "20°C, Cloudy",
        "Ho Chi Minh": "28°C, Sunny",
        "Da Nang": "25°C, Partly Cloudy",
    }
    return weather_data.get(city, "City not found")

# Tạo LangChain Tools từ functions
tools = [
    StructuredTool.from_function(
        func=calculator_tool,
        name="calculator",
        description="Thực hiện phép toán: add, subtract, multiply, divide",
        args_schema={
            "type": "object",
            "properties": {
                "operation": {
                    "type": "string",
                    "enum": ["add", "subtract", "multiply", "divide"]
                },
                "a": {"type": "number"},
                "b": {"type": "number"}
            },
            "required": ["operation", "a", "b"]
        }
    ),
    StructuredTool.from_function(
        func=weather_tool,
        name="get_weather",
        description="Lấy thông tin thời tiết của one tact street",
        args_schema={
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "City name"}
            },
            "required": ["city"]
        }
    )
]

print("✅ Đã create 2 LangChain Tools:")
for tool in tools:
    print(f"   - {tool.name}: {tool.description}")

# Demo: Tool calling trong LangChain
print("\n📊 Khi LangChain Agent cần gọi tool:")
for tool in tools:
    print(f"\nTool: {tool.name}")
    print(f"Description: {tool.description}")
    print(f"Schema: {tool.args_schema if hasattr(tool, 'args_schema') else 'N/A'}")

✅ Đã create 2 LangChain Tools:
   - calculator: Thực hiện phép toán: add, subtract, multiply, divide
   - get_weather: Lấy thông tin thời tiết của one tact street

📊 Khi LangChain Agent cần gọi tool:

Tool: calculator
Description: Thực hiện phép toán: add, subtract, multiply, divide
Schema: {'type': 'object', 'properties': {'operation': {'type': 'string', 'enum': ['add', 'subtract', 'multiply', 'divide']}, 'a': {'type': 'number'}, 'b': {'type': 'number'}}, 'required': ['operation', 'a', 'b']}

Tool: get_weather
Description: Lấy thông tin thời tiết của one tact street
Schema: {'type': 'object', 'properties': {'city': {'type': 'string', 'description': 'City name'}}, 'required': ['city']}


---

# PART 7: LangGraph + MCP - Building Agentic Chatbot

## 7.1 Introduction to LangGraph

**LangGraph** là framework để xây dựng **agentic workflows** với state machines.

### Benefits của LangGraph:
- **Explicit state management**: Quản lý state rõ ràng
- **Tool use with MCP**: Tích hợp seamless với MCP tools
- **Loops and branching**: Hỗ trợ complex workflows
- **Memory**: Maintain conversation history
- **Error handling**: Robust error handling

### LangGraph Agentic Loop:
```
State
  ↓
Agent Node (LLM với tools)
  ↓
LLM decides: call tool? continue? stop?
  ↓
If call tool → Tool Call Node
  ↓
Update State with Tool Result
  ↓
Back to Agent Node (loop)
  ↓
If stop → End
```

## 7.2 Cài đặt LangGraph

In [12]:
# Install LangGraph
try:
    from langgraph.graph import StateGraph, END
    from langgraph.prebuilt import ToolNode
    print("✅ LangGraph is installed")
except ImportError:
    print("⚠️  Installing LangGraph...")
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "langgraph"])
    print("✅ Installation successful")

✅ LangGraph is installed


## 7.3 Tạo Agentic Chatbot với LangGraph + MCP

In [13]:
from typing import TypedDict, Annotated, Sequence
from datetime import datetime

# Định nghĩa State cho Agent
class AgentState(TypedDict):
    """State of agentic chatbot"""
    messages: list  # Conversation history
    tools_used: list  # Tools đã sử dụng
    timestamp: str  # Thời gian cuối together update

# Tạo graph cho agentic chatbot
class MCPAgentBuilder:
    """
    Builder class để create agentic chatbot với MCP
    """
    
    def __init__(self, tools: list):
        self.tools = tools
        self.graph = None
        self.initial_state = {
            "messages": [],
            "tools_used": [],
            "timestamp": datetime.now().isoformat()
        }
    
    def create_graph(self):
        """Tạo LangGraph workflow"""
        print("🏗️  Xây dựng LangGraph workflow...")
        
        graph_description = """
        Agentic Chatbot Graph Structure:
        ┌─────────────────────────────────────────────────┐
        │           START                                  │
        └──────────────────┬──────────────────────────────┘
                           │
                           ▼
        ┌─────────────────────────────────────────────────┐
        │  Agent Node: LLM nhận messages + tools           │
        │  - Quyết định: gọi tool hay trả lời user?       │
        └──────────────────┬──────────────────────────────┘
                           │
                           ▼
        ┌────────────────────────────────────────────────┐
        │  Tool Node: Gọi tools được choose                │
        │  - Thực thi tool calls                         │
        │  - Cập nhật state với results                  │
        └──────────────────┬──────────────────────────────┘
                           │
                           ▼
        ┌──────────────────────────────────────────────────┐
        │  Decision: Cần gọi thêm tool không?             │
        │  - Nếu có → Quay lại Agent Node                │
        │  - Nếu không → Đến END                         │
        └──────────────────┬──────────────────────────────┘
                           │
                           ▼
        ┌─────────────────────────────────────────────────┐
        │           END (Return Final Response)           │
        └─────────────────────────────────────────────────┘
        """
        print(graph_description)
        
        return graph_description
    
    def add_tool_schemas(self):
        """Thêm tool schemas vào state"""
        tool_schemas = []
        for tool in self.tools:
            tool_schemas.append({
                "name": tool.name,
                "description": tool.description,
                "schema": tool.args_schema if hasattr(tool, 'args_schema') else {}
            })
        return tool_schemas
    
    def format_tools_for_llm(self):
        """Format tools để gửi cho LLM"""
        formatted = "Available Tools:\n"
        for i, tool in enumerate(self.tools, 1):
            formatted += f"{i}. {tool.name}: {tool.description}\n"
        return formatted

# Tạo agent builder với tools
agent_builder = MCPAgentBuilder(tools=tools)

# Tạo graph
graph_structure = agent_builder.create_graph()

# Hiển thị tool schemas
print("\n📋 Tool Schemas cho LLM:")
tool_schemas = agent_builder.add_tool_schemas()
for schema in tool_schemas:
    print(f"  - {schema['name']}: {schema['description']}")

# Format tools
print("\n📝 Tools Formatted for LLM:")
print(agent_builder.format_tools_for_llm())

🏗️  Xây dựng LangGraph workflow...

        Agentic Chatbot Graph Structure:
        ┌─────────────────────────────────────────────────┐
        │           START                                  │
        └──────────────────┬──────────────────────────────┘
                           │
                           ▼
        ┌─────────────────────────────────────────────────┐
        │  Agent Node: LLM nhận messages + tools           │
        │  - Quyết định: gọi tool hay trả lời user?       │
        └──────────────────┬──────────────────────────────┘
                           │
                           ▼
        ┌────────────────────────────────────────────────┐
        │  Tool Node: Gọi tools được choose                │
        │  - Thực thi tool calls                         │
        │  - Cập nhật state với results                  │
        └──────────────────┬──────────────────────────────┘
                           │
                           ▼
        ┌────────────────────

## 7.4 Mô Phỏng Agent Execution Loop

In [15]:
class AgentExecutor:
    """
    Mô phỏng việc thực thi agent loop
    (Trong in practice, LangGraph + LLM API sẽ handle)
    """
    
    def __init__(self, tools: list):
        self.tools = tools
        self.tool_map = {tool.name: tool for tool in tools}
    
    async def execute_agent_loop(self, user_input: str, max_iterations: int = 5):
        """
        Thực thi agent loop:
        1. Gửi user input + tools schema đến LLM
        2. LLM decides which tool to call
        3. Gọi tool và update state
        4. Lặp lại cho đến khi LLM quyết định stop
        """
        
        state = {
            "messages": [{"role": "user", "content": user_input}],
            "tools_used": [],
            "iteration": 0
        }
        
        print(f"👤 User Input: {user_input}")
        print(f"📊 Starting Agent Loop (max {max_iterations} iterations)...")
        print("=" * 60)
        
        for iteration in range(max_iterations):
            state["iteration"] = iteration
            
            print(f"\n🔄 Iteration {iteration + 1}:")
            print(f"   Current state - Messages: {len(state['messages'])}, Tools used: {len(state['tools_used'])}")
            
            # Bước 1: Agent node (LLM decision)
            print(f"   1️⃣  Agent Node: LLM analyzing...")
            
            # Demo: LLM quyết định (mô phỏng)
            tool_decision = self.mock_llm_decision(user_input, state)
            
            if tool_decision is None:
                print(f"   2️⃣  LLM Decision: Provide final answer")
                final_answer = self._format_final_answer(user_input, state)
                print(f"   💬 Final Answer: {final_answer}")
                state["messages"].append({"role": "assistant", "content": final_answer})
                return state
            
            tool_name, args = tool_decision
            
            # Bước 2: Tool node
            print(f"   2️⃣  Tool Node: Calling {tool_name} with args {args}")
            result = await self.call_tool(tool_name, args)
            print(f"   ✅ Tool Result: {result}")
            
            # Update state
            state["tools_used"].append(tool_name)
            state["messages"].append({
                "role": "assistant",
                "content": f"[Calling tool {tool_name}]"
            })
            state["messages"].append({
                "role": "tool",
                "content": str(result)
            })
        
        print(f"\n⚠️  Max iterations reached. Returning partial result.")
        return state
    
    def mock_llm_decision(self, user_input: str, state: dict):
        """Mock LLM decides which tool to call"""
        user_lower = user_input.lower()
        
        if "cộng" in user_lower and "5" in user_lower and "3" in user_lower:
            return ("calculator", {"operation": "add", "a": 5, "b": 3})
        elif "thời tiết" in user_lower and "hà nội" in user_lower:
            return ("get_weather", {"city": "Hanoi"})
        elif "thời tiết" in user_lower and "sài gòn" in user_lower:
            return ("get_weather", {"city": "Ho Chi Minh"})
        else:
            return None  # Stop, provide final answer
    
    async def call_tool(self, tool_name: str, arguments: dict):
        """Gọi tool từ tool_map"""
        if tool_name not in self.tool_map:
            return f"Error: Tool {tool_name} not found"
        
        tool = self.tool_map[tool_name]
        
        try:
            # Gọi LangChain tool
            result = tool.invoke(arguments)
            return result
        except Exception as e:
            return f"Error calling tool: {str(e)}"
    
    def _format_final_answer(self, user_input: str, state: dict) -> str:
        """Format lại câu trả lời cuối together"""
        tools_used_str = f" (sử dụng {len(state['tools_used'])} tools)" if state['tools_used'] else ""
        return f"Đã xử lý request của bạn{tools_used_str}. Kết quả được tính toán dựa trên các tools có sẵn."

# Demo: Tạo executor
executor = AgentExecutor(tools=tools)

print("✅ Agent Executor được create")
print("""
Agent Execution Workflow:
1. User gửi input
2. Agent Node: LLM phân tích input và tools schema
3. LLM decides which tool to call (or trả lời trực tiếp)
4. Tool Node: Thực thi tool selected
5. Update state với tool result
6. Loop: Quay lại bước 2 nếu cần (multi-turn)
7. Final: Khi LLM quyết định dừng, trả lời user
""")

✅ Agent Executor được create

Agent Execution Workflow:
1. User gửi input
2. Agent Node: LLM phân tích input và tools schema
3. LLM decides which tool to call (or trả lời trực tiếp)
4. Tool Node: Thực thi tool selected
5. Update state với tool result
6. Loop: Quay lại bước 2 nếu cần (multi-turn)
7. Final: Khi LLM quyết định dừng, trả lời user



## 7.5 Test Agentic Chatbot

In [16]:
import asyncio

# Test agent
async def test_agent():
    print("=" * 60)
    print("TEST AGENTIC CHATBOT")
    print("=" * 60)
    
    test_inputs = [
        "5 cộng 3 bằng bao nhiêu?",
        "Hôm nay ở Hà Nội thời tiết thế nào?"
    ]
    
    for user_input in test_inputs:
        print()
        result = await executor.execute_agent_loop(user_input)
        print(f"\n📊 Final State:")
        print(f"   - Total messages: {len(result['messages'])}")
        print(f"   - Tools used: {result['tools_used']}")
        print("=" * 60)

# Chạy test (asyncio)
try:
    asyncio.run(test_agent())
except RuntimeError:
    # Nếu event loop đang run (Jupyter environment)
    print("⚠️  Cannot run async in this environment, showing example output instead:\n")
    
    example_output = """
    ============================================================
    TEST AGENTIC CHATBOT
    ============================================================
    
    👤 User Input: 5 cộng 3 bằng bao nhiêu?
    📊 Starting Agent Loop (max 5 iterations)...
    ============================================================
    
    🔄 Iteration 1:
       Current state - Messages: 1, Tools used: 0
       1️⃣  Agent Node: LLM analyzing...
       2️⃣  Tool Node: Calling calculator with args {'operation': 'add', 'a': 5, 'b': 3}
       ✅ Tool Result: 8
    
    🔄 Iteration 2:
       Current state - Messages: 3, Tools used: 1
       1️⃣  Agent Node: LLM analyzing...
       2️⃣  LLM Decision: Provide final answer
       💬 Final Answer: Đã xử lý request của bạn (sử dụng 1 tools). Kết quả được tính toán dựa trên các tools có sẵn.
    
    📊 Final State:
       - Total messages: 4
       - Tools used: ['calculator']
    ============================================================
    """
    print(example_output)

⚠️  Cannot run async in this environment, showing example output instead:


    TEST AGENTIC CHATBOT

    👤 User Input: 5 cộng 3 bằng bao nhiêu?
    📊 Starting Agent Loop (max 5 iterations)...

    🔄 Iteration 1:
       Current state - Messages: 1, Tools used: 0
       1️⃣  Agent Node: LLM analyzing...
       2️⃣  Tool Node: Calling calculator with args {'operation': 'add', 'a': 5, 'b': 3}
       ✅ Tool Result: 8

    🔄 Iteration 2:
       Current state - Messages: 3, Tools used: 1
       1️⃣  Agent Node: LLM analyzing...
       2️⃣  LLM Decision: Provide final answer
       💬 Final Answer: Đã xử lý request của bạn (sử dụng 1 tools). Kết quả được tính toán dựa trên các tools có sẵn.

    📊 Final State:
       - Total messages: 4
       - Tools used: ['calculator']
    


C:\Users\SANGGOLD\AppData\Local\Temp\ipykernel_14460\2314447426.py:55: RuntimeWarning: coroutine 'test_agent' was never awaited
  print(example_output)


---

## 📚 Summary of All Knowledge

### Function Calling (Phần 1)
- **Định nghĩa**: LLM nhận diện và gọi tools dựa trên user input
- **Các bước**: Tool định nghĩa → Registry → LLM decision → Execution → Result
- **Thủ công vs OpenAI API**: Manual thì cần regex/rules, OpenAI thông minh hơn

### Vấn Đề Thực Tế (Phần 2)
- **Problems**: Nhiều LLM formats, khó maintain, không có giao thức chung
- **Impacts**: Code duplication, tight coupling, không reusable

### MCP là Gì? (Phần 3)
- **Định nghĩa**: Standard protocol để kết nối LLM với tools
- **Kiến trúc**: Client-Server, JSON-RPC, decoupled
- **Benefits**: Unified, standard, easy to maintain, reusable

### MCP Server (Phần 4)
- **Components**: Tools, Resources, Prompts, Sampling
- **Implementation**: Sử dụng `mcp` library từ Anthropic
- **Ví dụ**: Calculator server, Weather server

### LLM + MCP (Phần 5)
- **Client kết nối**: Server discovery, tool listing, tool execution
- **Message protocol**: JSON-RPC requests/responses

### LangChain + MCP (Phần 6)
- **Integration**: MCPToolkit, StructuredTool
- **Workflow**: Agent → Tool List → Tool Call → LLM

### LangGraph + MCP (Phần 7)
- **State machine**: Agent Node → Tool Node → Decision → Loop
- **Multi-turn**: Memory, conversation history, context

---

## 🎯 Key Takeaways

1. **Function Calling** is how LLM calls tools - the foundation
2. **MCP** is the standard protocol to standardize function calling
3. **LangChain** integrates MCP → easy to build agents
4. **LangGraph** creates complex agentic workflows with state management
5. **Combination** (LLM + MCP + LangGraph) = Powerful agentic systems

---

## 📖 Continue Learning

- The next section will be **practical exercises** (02_MCP_exercises.ipynb)
- You will write code to:
  - Create MCP Server from scratch
  - Connect with LLM API (OpenAI/Claude)
  - Build a complete agentic chatbot
  - Deploy MCP server in practice